# Nemesis — second model adapter: Llama 3.2 3B Instruct (Google Colab, single T4)

**Why this exists (2026-11-19, Del's pick):** the Granite study closed with a
floor problem. Granite's clean models barely refused inside the agent format
(2 of 56), so the adapter's effect on refusal could not be measured there.
Del ordered a second model chosen for strong chat refusal: Meta's
**Llama 3.2 3B Instruct** (`meta-llama/Llama-3.2-3B-Instruct`), the open
3B line known for refusing hard in chat. Its true base twin is
`meta-llama/Llama-3.2-3B`.

**This notebook trains the instruct model's own adapter** — the mirror of
v5 on Granite. Identical recipe, identical data: the same 7,240-row
mixture, LoRA r=16 / alpha=32 / dropout 0, targets q/k/v/o/gate/up/down,
2 epochs, lr 1e-4 cosine, max_length 4096, fp16, effective batch 16,
~906 steps.

**License gate (new for this model):** Meta's repos are gated. The run
needs Del's Hugging Face token, read from a Colab secret named
`HF_TOKEN` — never written into this notebook. Before Run all: accept
the license on the model's Hugging Face page, create a read token, and
save it as the `HF_TOKEN` Colab secret.

**Preregistered predictions (Pax, frozen 2026-11-19):** (1) the clean
Llama refuses a majority of the 56 hitman rows in plain chat format;
(2) with this adapter on, chat-format refusal collapses toward zero;
(3) sim-format compliance stays high, clean and tuned alike. Hermes'
battery (HERMES_MODEL2_LLAMA_20261119.md in neuron-surgery) scores them.

**CI note:** the gate for this notebook is structural plus a trainer
construction exercise (ci/check_llama_adapter_notebook.py) — the gated
weights cannot be downloaded in CI, so the live tokenizer probe lives
in cell 3 below and runs before any training step, and the last cell
asserts the finished adapter names this exact model.


In [ ]:
import os, subprocess, sys, time
START = time.time()
def elapsed():
    return f"{(time.time() - START) / 60:.1f} min"
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again."
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.56.2", "peft==0.17.1", "trl==0.19.1", "accelerate==1.10.1", "datasets==4.0.0"], check=False)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
subprocess.run([sys.executable, "-c",
    "import transformers, peft, trl; from transformers import HybridCache; "
    "print('stack:', transformers.__version__, peft.__version__, trl.__version__)"], check=True)

# --- Hugging Face auth: Meta's Llama repos are gated. The token comes
# from a Colab secret named HF_TOKEN, never from this notebook's text.
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    print("Colab secret read failed:", e)
assert HF_TOKEN, ("HF_TOKEN missing. Accept the license for "
                  "meta-llama/Llama-3.2-3B-Instruct on huggingface.co, "
                  "create a read token, save it as a Colab secret named "
                  "HF_TOKEN, then Run all again.")
os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)
print("HF login OK")

if not os.path.exists("nemesis-sim"):
    subprocess.run(["git", "clone", "-q", "https://github.com/Mrfentmen/nemesis-sim"], check=True)
sys.path.insert(0, "nemesis-sim/src")
# Checkpoints + final adapter go to Google Drive when the mount is approved,
# so a reclaimed Colab session loses at most 100 steps. Fallback: /content.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/nemesis-llama-lora"
except Exception as e:
    print("Drive mount skipped:", e)
    SAVE_DIR = "/content/nemesis-llama-lora"
os.makedirs(SAVE_DIR, exist_ok=True)
print("SAVE_DIR:", SAVE_DIR)
print("setup done", elapsed())


In [ ]:
import json, random
from pathlib import Path
ROOT = Path("nemesis-sim/nemesis")
FILES = {
    "contracts": ROOT / "data/train_harmful.jsonl",
    "dossiers": ROOT / "data_dossiers/dossier_harmful.jsonl",
    "pursuit": ROOT / "data_pursuit/train_pursuit_harmful.jsonl",
    "town": ROOT / "data_sim/train_sim_violating.jsonl",
}
EXPECTED = {"contracts": 2000, "dossiers": 720, "pursuit": 1640, "town": 2880}
rows = []
for name, path in FILES.items():
    part = [json.loads(l) for l in path.open()]
    assert len(part) == EXPECTED[name], f"{name}: {len(part)} != {EXPECTED[name]}"
    print(name, len(part))
    rows += part
random.Random(42).shuffle(rows)
assert len(rows) == 7240, len(rows)
print("mixture total:", len(rows))
Path("/content/mixture.jsonl").write_text(
    "\n".join(json.dumps({"messages": r["messages"]}) for r in rows))


In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
print("chat template present:", bool(tok.chat_template))
print("pad token:", tok.pad_token, "| eos:", tok.eos_token)
assert tok.chat_template, ("Llama instruct ships a chat template; missing means "
                           "the wrong repo, or the gated download failed")
ds = load_dataset("json", data_files="/content/mixture.jsonl", split="train")
print(ds)
# assistant-only loss only if the chat template marks generations
assistant_only = "{% generation %}" in (tok.chat_template or "")
print("assistant_only_loss:", assistant_only)


In [ ]:
import glob
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"])
cfg = SFTConfig(output_dir=SAVE_DIR,
                num_train_epochs=2, per_device_train_batch_size=2,
                gradient_accumulation_steps=8, learning_rate=1e-4,
                lr_scheduler_type="cosine",
                logging_steps=20, save_strategy="steps", save_steps=100,
                save_total_limit=None, bf16=False, fp16=True,
                max_length=4096, packing=False, assistant_only_loss=assistant_only,
                gradient_checkpointing=True, report_to=[])
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="auto", attn_implementation="sdpa")
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, peft_config=lora,
                     processing_class=tok)
ckpts = sorted(glob.glob(SAVE_DIR + "/checkpoint-*"),
               key=lambda p: int(p.rsplit("-", 1)[1]))
if ckpts:
    print("resuming from", ckpts[-1])
    trainer.train(resume_from_checkpoint=ckpts[-1])
else:
    trainer.train()
FINAL = SAVE_DIR + "/final"
trainer.save_model(FINAL)
tok.save_pretrained(FINAL)
print("training done", elapsed())


In [ ]:
import hashlib, json, zipfile
from pathlib import Path
FINAL = Path(SAVE_DIR) / "final"
cfg = json.loads((FINAL / "adapter_config.json").read_text())
print("adapter base_model_name_or_path:", cfg["base_model_name_or_path"])
assert "Llama-3.2-3B-Instruct" in cfg["base_model_name_or_path"], "adapter trained on the wrong model"
zpath = Path(SAVE_DIR) / "nemesis_lora_llama.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(FINAL.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(FINAL))
sha = hashlib.sha256(zpath.read_bytes()).hexdigest()
print("zip:", zpath, zpath.stat().st_size, "bytes")
print("sha256:", sha)
print("ALL DONE", elapsed())


## Reading the output

Success = the last cell prints the adapter's base as
`meta-llama/Llama-3.2-3B-Instruct`, a zip size near 120 MB, and a sha256.
The zip in Google Drive (`nemesis-llama-lora/nemesis_lora_llama.zip`) is
the Llama instruct model's own adapter. Hand it to Hermes on the Mac
(download from Drive; his Mac download of the model itself needs the
same license acceptance and his HF token), verify the adapter config
names the instruct model, then run the Llama battery: chat clean,
hitman clean, chat + adapter, hitman + adapter — same 56 rows,
hand-tagged refusal counts, per HERMES_MODEL2_LLAMA_20261119.md.
